# Stability-Weighted Near-Zero Fine-Tune

Notebook này kiểm tra giả thuyết:

- near-zero label volatility là một root cause thật sự
- dùng trọng số ổn định trên vùng `|y| <= 0.2` có thể giảm failure mode nguy hiểm
- chưa cần full production retrain; chỉ cần fine-tune ngắn từ `ckpt_best.pt`

Tất cả artifact được lưu trong:
`C:\Users\USER\Desktop\chess_engine\experiments\stability_weighted_near_zero_finetune\outputs`

In [1]:
from dataclasses import asdict
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path(r"C:\Users\USER\Desktop\chess_engine")
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "stability_weighted_near_zero_finetune"
RUN_DIR = Path(r"C:\Users\USER\Downloads\dgrn_5m_v3_stage2_polish_run1")
DATA_ROOT = PROJECT_ROOT / "data" / "process"

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import stability_weighted_helpers as lab

torch.set_float32_matmul_precision("high")
lab.set_global_seed(123)
paths = lab.build_default_paths(run_dir=RUN_DIR, data_root=DATA_ROOT, experiment_dir=EXPERIMENT_DIR)
lab.export_paths_json(paths, paths["output_dir"] / "paths.json")

if "envs\\chess_engine" not in sys.executable.lower():
    raise RuntimeError(
        f"Notebook is running under the wrong interpreter: {sys.executable}. "
        "Select the 'chess_engine' Jupyter kernel."
    )

DEVICE = lab.choose_device(prefer_cuda=True)
if DEVICE.type != "cuda":
    raise RuntimeError("CUDA is required for this notebook.")

CHECKPOINTS = {
    "best": paths["run_dir"] / "ckpt_best.pt",
    "latest": paths["run_dir"] / "ckpt_latest.pt",
}
for name, ckpt_path in CHECKPOINTS.items():
    assert ckpt_path.exists(), f"Missing checkpoint: {name} -> {ckpt_path}"

WEIGHT_CFG = lab.StabilityWeightConfig(
    near_zero_thr=0.20,
    calibration_abs_y_sample_edges=(0.0, 0.05, 0.10, 0.15, 0.20),
    sample_per_abs_y_band=24,
    stockfish_path=r"D:\stockfish-windows-x86-64-avx2\stockfish\stockfish-windows-x86-64-avx2.exe",
    stockfish_threads=1,
    stockfish_hash_mb=32,
    stockfish_node_budgets=(2_000, 8_000, 32_000),
    stockfish_command_pause_ms=50,
    stockfish_timeout_sec=15.0,
    calibration_seed=123,
    prediction_batch_size=2560,
    weight_abs_y_edges=(0.0, 0.025, 0.05, 0.10, 0.15, 0.20),
    teacher_abs_err_quantiles=(0.2, 0.4, 0.6, 0.8),
    smoothing_prior=3.0,
    weight_strength=0.45,
    weight_min=0.55,
)

TRAIN_CFG = lab.FineTuneConfig(
    lambda_y=0.99,
    z_loss_beta=1.0,
    z_huber_delta=0.5,
    target_clamp_eps=1e-3,
    learning_rate=3e-6,
    min_lr=1e-6,
    weight_decay=2e-4,
    grad_clip_norm=1.0,
    batch_size=640,
    epochs=1,
    log_every_steps=200,
    seed=123,
    eval_val_samples=100_000,
    eval_test_samples=200_000,
    eval_val_num_shards=2,
    eval_test_num_shards=4,
    train_num_shards=None,
)

NOTEBOOK_CONFIG = {
    "weight_cfg": asdict(WEIGHT_CFG),
    "train_cfg": asdict(TRAIN_CFG),
    "prediction_cache_split": "train",
}
assert Path(WEIGHT_CFG.stockfish_path).exists(), f"Missing Stockfish binary: {WEIGHT_CFG.stockfish_path}"
weight_cfg_validation = lab.validate_stability_weight_config(WEIGHT_CFG)
train_cfg_validation = lab.validate_finetune_config(TRAIN_CFG)
lab.save_json(NOTEBOOK_CONFIG, paths["output_dir"] / "runtime_config.json")
lab.save_json(weight_cfg_validation, paths["reports_dir"] / "weight_cfg_validation.json")
lab.save_json(train_cfg_validation, paths["reports_dir"] / "train_cfg_validation.json")
print("python:", sys.executable)
print("device:", DEVICE)
print("gpu_name:", torch.cuda.get_device_name(0))
display(pd.DataFrame({"path_key": list(paths.keys()), "path_value": [str(v) for v in paths.values()]}))
display(pd.DataFrame([weight_cfg_validation]))
display(pd.DataFrame([train_cfg_validation]))

python: c:\Users\USER\anaconda3\envs\chess_engine\python.exe
device: cuda
gpu_name: NVIDIA GeForce RTX 2050


,path_key,path_value
0,project_root,C:\Users\USER\Desktop\chess_engine
1,run_dir,C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...
2,data_root,C:\Users\USER\Desktop\chess_engine\data\process
3,experiment_dir,C:\Users\USER\Desktop\chess_engine\experiments...
4,output_dir,C:\Users\USER\Desktop\chess_engine\experiments...
5,plots_dir,C:\Users\USER\Desktop\chess_engine\experiments...
6,reports_dir,C:\Users\USER\Desktop\chess_engine\experiments...
7,checkpoints_dir,C:\Users\USER\Desktop\chess_engine\experiments...
8,cache_dir,C:\Users\USER\Desktop\chess_engine\experiments...
9,train_pred_cache_dir,C:\Users\USER\Desktop\chess_engine\experiments...


,is_valid,near_zero_thr,calibration_abs_y_sample_edges,weight_abs_y_edges,stockfish_node_budgets,teacher_abs_err_quantiles,issues
0,True,0.2,"[0.0, 0.05, 0.1, 0.15, 0.2]","[0.0, 0.025, 0.05, 0.1, 0.15, 0.2]","[2000, 8000, 32000]","[0.2, 0.4, 0.6, 0.8]",[]


,is_valid,issues,batch_size,epochs,learning_rate,min_lr
0,True,[],640,1,0.000003,0.000001


## Runtime Self-Check

Cell này xác minh:
- batch size có thực tế trên GPU hiện tại hay không
- rough runtime cho 1 epoch full dataset
- decode -> FEN có hợp lệ cho Stockfish hay không
- Stockfish proxy có deterministic ở mức self-check hay không
- checkpoint config và dataset layout có khớp expectation hay không

In [2]:
benchmark = lab.benchmark_single_train_step(
    init_ckpt_path=CHECKPOINTS["best"],
    data_root=paths["data_root"],
    device=DEVICE,
    batch_size=TRAIN_CFG.batch_size,
    num_shards=TRAIN_CFG.train_num_shards,
)
lab.save_json(benchmark, paths["reports_dir"] / "runtime_benchmark.json")
stockfish_benchmark = lab.benchmark_stockfish_proxy(WEIGHT_CFG)
lab.save_json(stockfish_benchmark, paths["reports_dir"] / "stockfish_proxy_benchmark.json")
roundtrip = lab.base_lab.validate_encode_decode_roundtrip(
    data_root=paths["data_root"],
    split="train",
    sample_count=64,
)
lab.save_json(roundtrip, paths["reports_dir"] / "encode_decode_roundtrip.json")
stockfish_decode = lab.validate_stockfish_compatible_decoding(
    data_root=paths["data_root"],
    split="train",
    sample_count=64,
    num_shards=1,
)
lab.save_json(stockfish_decode, paths["reports_dir"] / "stockfish_decode_validation.json")
stockfish_validation = lab.validate_stockfish_proxy(WEIGHT_CFG)
lab.save_json(stockfish_validation, paths["reports_dir"] / "stockfish_proxy_validation.json")
stockfish_dataset_validation = lab.validate_stockfish_proxy_on_dataset_sample(
    data_root=paths["data_root"],
    split="train",
    cfg=WEIGHT_CFG,
    sample_index=0,
    num_shards=1,
)
lab.save_json(stockfish_dataset_validation, paths["reports_dir"] / "stockfish_proxy_dataset_validation.json")

checkpoint_rows = []
for label, ckpt_path in CHECKPOINTS.items():
    payload = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    checkpoint_rows.append(
        {
            "label": label,
            "epoch": payload.get("epoch"),
            "loss_mode": payload.get("loss_mode"),
            "output_mode": payload.get("output_mode"),
            "y_loss_weight_start": payload.get("y_loss_weight_start"),
            "y_loss_weight_end": payload.get("y_loss_weight_end"),
            "y_loss_ramp_epochs": payload.get("y_loss_ramp_epochs"),
            "z_loss_beta": payload.get("z_loss_beta"),
            "z_huber_delta": payload.get("z_huber_delta"),
            "lr": payload.get("lr"),
        }
    )
checkpoint_df = pd.DataFrame(checkpoint_rows)
lab.save_dataframe(checkpoint_df, paths["reports_dir"] / "checkpoint_config_table.csv")

split_summary = pd.DataFrame(
    [
        lab.summarize_split_layout(paths["data_root"], "train", num_shards=TRAIN_CFG.train_num_shards),
        lab.summarize_split_layout(paths["data_root"], "val"),
        lab.summarize_split_layout(paths["data_root"], "test"),
    ]
)
lab.save_dataframe(split_summary, paths["reports_dir"] / "split_summary.csv")
assert roundtrip["mismatches"] == 0, roundtrip
if stockfish_decode["checked_with_python_chess"]:
    assert stockfish_decode["invalid_fens"] == 0, stockfish_decode
assert stockfish_validation["bestmove_match"], stockfish_validation
assert stockfish_validation["target_match"], stockfish_validation
assert stockfish_dataset_validation["bestmove_match"], stockfish_dataset_validation
assert stockfish_dataset_validation["target_match"], stockfish_dataset_validation
display(pd.DataFrame([benchmark]))
display(pd.DataFrame(stockfish_benchmark["per_query"]))
display(pd.DataFrame([{"one_position_total_sec": stockfish_benchmark["one_position_total_sec"], "estimated_positions": stockfish_benchmark["estimated_positions"], "estimated_total_min": stockfish_benchmark["estimated_total_min"]}]))
display(pd.DataFrame([roundtrip]))
display(pd.DataFrame([stockfish_decode]))
display(pd.DataFrame([stockfish_validation]))
display(pd.DataFrame([stockfish_dataset_validation]))
display(checkpoint_df)
display(split_summary)

,batch_size,step_time_sec,peak_mem_gb,steps_per_epoch,train_total_samples,train_num_shards,epoch_hours_estimate
0,640,1.454446,3.600463,6250,4000000,80,2.52508


,node_budget,elapsed_sec,target_value,bestmove
0,2000,0.458819,0.076517,e2e4
1,8000,0.459420,0.088104,e2e4
2,32000,0.460674,0.061589,e2e4


,one_position_total_sec,estimated_positions,estimated_total_min
0,1.378912,96,2.20626


,sample_count,mismatches,mismatch_examples
0,64,0,[]


,sample_count,invalid_fens,checked_with_python_chess,examples
0,64,0,True,[]


,probe_fen,node_budgets,first_bestmoves,second_bestmoves,first_targets,second_targets,bestmove_match,target_match
0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,"[2000, 8000, 32000]","[e2e4, e2e4, e2e4]","[e2e4, e2e4, e2e4]","[0.07651680911202627, 0.08810429968944254, 0.0...","[0.07651680911202627, 0.08810429968944254, 0.0...",True,True


,probe_fen,node_budgets,first_bestmoves,second_bestmoves,first_targets,second_targets,bestmove_match,target_match,split,sample_index
0,r3kb1r/p1n1qpp1/2p1p3/1p2Pn1p/3P3P/P2Q1N2/4NPP...,"[2000, 8000, 32000]","[e1g1, e2g3, e2g3]","[e1g1, e2g3, e2g3]","[-0.04164257074548595, 0.0066665679029903665, ...","[-0.04164257074548595, 0.0066665679029903665, ...",True,True,train,0


,label,epoch,loss_mode,output_mode,y_loss_weight_start,y_loss_weight_end,y_loss_ramp_epochs,z_loss_beta,z_huber_delta,lr
0,best,1,None,None,None,None,None,None,None,None
1,latest,2,None,None,None,None,None,None,None,None


,split,num_shards,samples
0,train,80,4000000
1,val,10,500000
2,test,10,500000


## Baseline Teacher Eval

Đo teacher hiện tại trên đúng metric suite sẽ dùng để gate fine-tune.

In [3]:
baseline_model, _ = lab.base_lab.load_model_from_checkpoint(CHECKPOINTS["best"], device=DEVICE)
baseline_val = lab.evaluate_model_on_split(
    model=baseline_model,
    data_root=paths["data_root"],
    split="val",
    device=DEVICE,
    max_samples=TRAIN_CFG.eval_val_samples,
    num_shards=TRAIN_CFG.eval_val_num_shards,
    batch_size=max(TRAIN_CFG.batch_size, 1024),
)
baseline_test = lab.evaluate_model_on_split(
    model=baseline_model,
    data_root=paths["data_root"],
    split="test",
    device=DEVICE,
    max_samples=TRAIN_CFG.eval_test_samples,
    num_shards=TRAIN_CFG.eval_test_num_shards,
    batch_size=max(TRAIN_CFG.batch_size, 1024),
)
del baseline_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
lab.save_json(baseline_val["metrics"], paths["reports_dir"] / "baseline_val_metrics.json")
lab.save_json(baseline_test["metrics"], paths["reports_dir"] / "baseline_test_metrics.json")
baseline_compare = pd.DataFrame(
    [
        lab.compare_metric_rows("baseline_val", baseline_val["metrics"]),
        lab.compare_metric_rows("baseline_test", baseline_test["metrics"]),
    ]
)
lab.save_dataframe(baseline_compare, paths["reports_dir"] / "baseline_compare.csv")
display(baseline_compare)

[eval_val] offset=0 / 100000 elapsed=3.8s
[eval_val] offset=25600 / 100000 elapsed=85.3s
[eval_val] offset=51200 / 100000 elapsed=169.9s
[eval_val] offset=76800 / 100000 elapsed=255.2s
[eval_test] offset=0 / 200000 elapsed=3.6s
[eval_test] offset=25600 / 200000 elapsed=91.0s
[eval_test] offset=51200 / 200000 elapsed=176.8s
[eval_test] offset=76800 / 200000 elapsed=265.0s
[eval_test] offset=102400 / 200000 elapsed=350.9s
[eval_test] offset=128000 / 200000 elapsed=437.4s
[eval_test] offset=153600 / 200000 elapsed=533.9s
[eval_test] offset=179200 / 200000 elapsed=626.0s


,label,overall_mse,overall_mae,mse_0.7,mae_0.7,slope_0.7,bias_0.7,mse_0.2,r2_0.2,false_0.1_0.3,false_0.2_0.4,center_spread_ratio_0.05,max_midband_abs_cal_gap,gate_score
0,baseline_val,0.066763,0.176674,0.050984,0.158812,0.608713,0.006135,0.030883,-4.044855,0.102807,0.053161,8.734883,0.263877,0.216178
1,baseline_test,0.067602,0.177554,0.051428,0.159380,0.605962,0.005631,0.031037,-4.076108,0.106200,0.053206,8.755165,0.266370,0.219630


## Build Stability Weights

Trình tự:
1. cache full-train teacher predictions
2. lấy calibration subset near-zero cân bằng theo `|y|`
3. chạy Stockfish subset oracle theo nhiều budget `nodes`
4. fit lookup table `weight(abs_y, teacher_abs_err)`

In [4]:
pred_cache = lab.precompute_train_prediction_cache(
    init_ckpt_path=CHECKPOINTS["best"],
    data_root=paths["data_root"],
    output_dir=paths["output_dir"],
    device=DEVICE,
    batch_size=WEIGHT_CFG.prediction_batch_size,
    num_shards=TRAIN_CFG.train_num_shards,
)

subset = lab.build_near_zero_calibration_subset(
    data_root=paths["data_root"],
    train_pred_cache_dir=paths["train_pred_cache_dir"],
    cfg=WEIGHT_CFG,
    num_shards=TRAIN_CFG.train_num_shards,
)
lab.save_dataframe(subset["count_table"], paths["reports_dir"] / "calibration_candidate_count_table.csv")
lab.save_dataframe(subset["quota_table"], paths["reports_dir"] / "calibration_quota_table.csv")
lab.save_dataframe(subset["quota_summary"], paths["reports_dir"] / "calibration_quota_summary.csv")
display(subset["quota_summary"])

proxy = lab.run_weight_calibration_stockfish_proxy(
    subset=subset["samples"],
    cfg=WEIGHT_CFG,
    output_dir=paths["output_dir"],
)
lookup = lab.build_stability_weight_lookup(
    calibration_rows=proxy["rows"],
    cfg=WEIGHT_CFG,
    output_dir=paths["output_dir"],
)
weight_audit = lab.audit_full_train_weight_distribution(
    data_root=paths["data_root"],
    train_pred_cache_dir=paths["train_pred_cache_dir"],
    lookup=lookup,
    cfg=WEIGHT_CFG,
    output_dir=paths["output_dir"],
    num_shards=TRAIN_CFG.train_num_shards,
)
lab.save_json(proxy["report"], paths["reports_dir"] / "weight_calibration_stockfish_report.json")
display(pd.DataFrame([proxy["report"]["stable"], proxy["report"]["unstable"]], index=["stable", "unstable"]))
display(pd.DataFrame([weight_audit["report"]]))
display(lookup["cell_table"].head(12))

[train_pred_cache_00000] offset=0 / 50000 elapsed=1.3s
[train-pred-cache] shards=1/80 elapsed=25.8s
[train_pred_cache_00001] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00002] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00003] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00004] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00005] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00006] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00007] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00008] offset=0 / 50000 elapsed=1.3s
[train-pred-cache] shards=9/80 elapsed=232.0s
[train_pred_cache_00009] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00010] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00011] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00012] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00013] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00014] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00015] offset=0 / 50000 elapsed=1.3s
[train_pred_cache_00016] offs

,band_idx,band_label,count,quota
0,0,"[0.000,0.050]",1257236,24
1,1,"[0.050,0.100]",462764,24
2,2,"[0.100,0.150]",280468,24
3,3,"[0.150,0.200]",199532,24


[weight-calibration-stockfish] processed=1/96
[weight-calibration-stockfish] processed=17/96
[weight-calibration-stockfish] processed=33/96
[weight-calibration-stockfish] processed=49/96
[weight-calibration-stockfish] processed=65/96
[weight-calibration-stockfish] processed=81/96


,n,mse,mae,false_decisive_0.3,mean_instability_score,mean_sf_target_range,mean_sf_target_std,mean_sf_bestmove_changes,mean_sf_sign_flips,mean_sf_final_gap_to_train_target
stable,32,0.024460,0.102069,0.09375,0.280263,0.02453,0.010496,0.5,0.15625,0.031587
unstable,32,0.024563,0.111061,0.15625,0.714309,0.08971,0.038956,1.0,0.18750,0.040115


,total_samples,total_near_zero_samples,near_zero_fraction,mean_weight_all,mean_weight_near_zero
0,4000000,2200000,0.55,0.871365,0.766117


,abs_y_bin_id,abs_y_left,abs_y_right,err_bin_id,err_left,err_right,count,raw_instability_mean,band_prior_instability,smoothed_instability,weight
0,0,0.000,0.025,0,0.000000,0.029038,4,0.417105,0.536842,0.468421,0.789211
1,0,0.000,0.025,1,0.029038,0.055772,5,0.506316,0.536842,0.517763,0.767007
2,0,0.000,0.025,2,0.055772,0.097473,5,0.645263,0.536842,0.604605,0.727928
3,0,0.000,0.025,3,0.097473,0.174841,5,0.580526,0.536842,0.564145,0.746135
4,0,0.000,0.025,4,0.174841,0.519044,4,0.504605,0.536842,0.518421,0.766711
5,1,0.025,0.050,0,0.000000,0.029038,0,NaN,0.568421,0.568421,0.744211
6,1,0.025,0.050,1,0.029038,0.055772,1,0.568421,0.568421,0.568421,0.744211
7,1,0.025,0.050,2,0.055772,0.097473,0,NaN,0.568421,0.568421,0.744211
8,1,0.025,0.050,3,0.097473,0.174841,0,NaN,0.568421,0.568421,0.744211
9,1,0.025,0.050,4,0.174841,0.519044,0,NaN,0.568421,0.568421,0.744211


## Stability-Weighted Fine-Tune

Đây là run kiểm tra giả thuyết chính:
- fine-tune từ `ckpt_best`
- dùng weighted hybrid objective trên full train split
- đánh giá lại bằng cùng metric suite

In [5]:
finetune = lab.run_stability_weighted_finetune(
    init_ckpt_path=CHECKPOINTS["best"],
    data_root=paths["data_root"],
    output_dir=paths["output_dir"],
    device=DEVICE,
    lookup=lookup,
    weight_cfg=WEIGHT_CFG,
    train_cfg=TRAIN_CFG,
)
display(finetune["history"])

[stability-ft] finished shard 1/80
[stability-ft] finished shard 2/80
[stability-ft][epoch=0] step=200/6250 obj=0.053604 weighted_mse=0.053903 plain_mse=0.050446
[stability-ft] finished shard 3/80
[stability-ft] finished shard 4/80
[stability-ft] finished shard 5/80
[stability-ft][epoch=0] step=400/6250 obj=0.053976 weighted_mse=0.054277 plain_mse=0.050802
[stability-ft] finished shard 6/80
[stability-ft] finished shard 7/80
[stability-ft][epoch=0] step=600/6250 obj=0.053996 weighted_mse=0.054297 plain_mse=0.050841
[stability-ft] finished shard 8/80
[stability-ft] finished shard 9/80
[stability-ft] finished shard 10/80
[stability-ft][epoch=0] step=800/6250 obj=0.053975 weighted_mse=0.054277 plain_mse=0.050818
[stability-ft] finished shard 11/80
[stability-ft] finished shard 12/80
[stability-ft][epoch=0] step=1000/6250 obj=0.053979 weighted_mse=0.054280 plain_mse=0.050838
[stability-ft] finished shard 13/80
[stability-ft] finished shard 14/80
[stability-ft] finished shard 15/80
[stabili

,epoch,train_objective,train_weighted_mse,train_weighted_z_huber,train_plain_mse,val_mse_0.7,val_slope_0.7,val_false_0.1_0.3,val_false_0.2_0.4,val_center_spread_ratio_0.05,val_max_midband_abs_cal_gap,gate_score,lr,epoch_time_sec
0,0,0.053804,0.054104,0.024166,0.050734,0.052062,0.633214,0.113011,0.060088,9.096658,0.251701,0.213189,0.000001,7351.623857


In [6]:
finetuned_val = lab.base_lab.run_teacher_eval_suite(
    ckpt_path=finetune["best_checkpoint"],
    data_root=paths["data_root"],
    split="val",
    device=DEVICE,
    output_dir=paths["output_dir"],
    max_samples=TRAIN_CFG.eval_val_samples,
    batch_size=max(TRAIN_CFG.batch_size, 1024),
    num_shards=TRAIN_CFG.eval_val_num_shards,
    prefix="stability_weighted_best_val",
)
finetuned_test = lab.base_lab.run_teacher_eval_suite(
    ckpt_path=finetune["best_checkpoint"],
    data_root=paths["data_root"],
    split="test",
    device=DEVICE,
    output_dir=paths["output_dir"],
    max_samples=TRAIN_CFG.eval_test_samples,
    batch_size=max(TRAIN_CFG.batch_size, 1024),
    num_shards=TRAIN_CFG.eval_test_num_shards,
    prefix="stability_weighted_best_test",
)

compare = pd.DataFrame(
    [
        lab.compare_metric_rows("baseline_test", baseline_test["metrics"]),
        lab.compare_metric_rows("stability_weighted_best_test", finetuned_test["metrics"]),
    ]
)
compare["delta_vs_baseline"] = compare["gate_score"] - compare["gate_score"].iloc[0]
lab.save_dataframe(compare, paths["reports_dir"] / "stability_weighted_test_compare.csv")
lab.save_json(finetuned_val["metrics"], paths["reports_dir"] / "stability_weighted_best_val_metrics.json")
lab.save_json(finetuned_test["metrics"], paths["reports_dir"] / "stability_weighted_best_test_metrics.json")
display(compare)

[stability_weighted_best:val] offset=0 / 100000 elapsed=0.6s
[stability_weighted_best:val] offset=25600 / 100000 elapsed=15.5s
[stability_weighted_best:val] offset=51200 / 100000 elapsed=30.5s
[stability_weighted_best:val] offset=76800 / 100000 elapsed=45.4s
[stability_weighted_best:test] offset=0 / 200000 elapsed=0.8s
[stability_weighted_best:test] offset=25600 / 200000 elapsed=15.7s
[stability_weighted_best:test] offset=51200 / 200000 elapsed=30.7s
[stability_weighted_best:test] offset=76800 / 200000 elapsed=45.7s
[stability_weighted_best:test] offset=102400 / 200000 elapsed=60.6s
[stability_weighted_best:test] offset=128000 / 200000 elapsed=75.6s
[stability_weighted_best:test] offset=153600 / 200000 elapsed=90.6s
[stability_weighted_best:test] offset=179200 / 200000 elapsed=105.5s


,label,overall_mse,overall_mae,mse_0.7,mae_0.7,slope_0.7,bias_0.7,mse_0.2,r2_0.2,false_0.1_0.3,false_0.2_0.4,center_spread_ratio_0.05,max_midband_abs_cal_gap,gate_score,delta_vs_baseline
0,baseline_test,0.067602,0.177554,0.051428,0.159380,0.605962,0.005631,0.031037,-4.076108,0.106200,0.053206,8.755165,0.266370,0.21963,0.00000
1,stability_weighted_best_test,0.067926,0.177979,0.052563,0.161296,0.630342,0.003417,0.033518,-4.481795,0.116427,0.059205,9.104031,0.254077,0.21671,-0.00292
